Autograd is automatic differentiation—the brain inside deep-learning frameworks (like PyTorch) that figures out how to compute gradients for you, automatically.

In [1]:
import torch

# 1.Calculating the most basic examples first:

## Example 1:calculating dy_dx of x^2:

In [3]:
x=torch.tensor(3.0,requires_grad=True) #requires grad parameter understands that we are going to calculate gradient of x. by default this parameter is false
y=x**2

In [4]:
x

tensor(3., requires_grad=True)

In [5]:
y

tensor(9., grad_fn=<PowBackward0>)

In [6]:
y.backward()

In [7]:
x.grad

tensor(6.)

1️⃣ y.backward()
What it means in simple words

“Go backwards and calculate gradients.”

When you call:

y.backward()


PyTorch:

Starts from y (usually loss)

Walks backward through the computation graph
Uses the chain rule
Computes derivatives with respect to all tensors that have
requires_grad=True
Think of it like this 🧠
Forward pass → compute output
Backward pass → compute how much each input affected the output
y.backward() = start backpropagation


Important rules about backward()
🔹 Rule 1: y must be a scalar

backward() works directly only on scalar values (like loss).

If not:

y.backward(torch.ones_like(y))

🔹 Rule 2: Graph is freed after backward

By default, PyTorch deletes the graph to save memory.

To keep it:

y.backward(retain_graph=True)



2️⃣ x.grad
What it means in simple words

“Give me the gradient of y with respect to x.”

After you run:

y.backward()


You can ask:

print(x.grad)

```
# This is formatted as code
```




This returns:

𝑑
𝑦
𝑑
𝑥
dx
dy
	​

Example walkthrough
x = torch.tensor(3.0, requires_grad=True)
y = x*x + 2*x

y.backward()
print(x.grad)


Output:

tensor(8.)


Because:2(3)+2=8



What exactly is stored in .grad

.grad stores the partial derivative

Shape is same as\ the tensor

Only exists for tensors with requires_grad=T

```
# This is formatted as code
```

rue


## Example 2.calculating dz_dx where y=x^2 and z=sin y:

In [21]:
x=torch.tensor(3.0,requires_grad=True)
x

tensor(3., requires_grad=True)

In [22]:
y=x**2
y

tensor(9., grad_fn=<PowBackward0>)

In [23]:
z=torch.sin(y)
z

tensor(0.4121, grad_fn=<SinBackward0>)

In [24]:
z.backward() #-> derivative is calculated

In [26]:
x.grad #-> gives the value of the derivate

tensor(-5.4668)

# 2. Calculating the derivates(gradients) for a neural network from scratch(without autograd)

In [27]:
import torch

# Inputs
x = torch.tensor(6.7)  # Input feature
y = torch.tensor(0.0)  # True label (binary)

w = torch.tensor(1.0)  # Weight
b = torch.tensor(0.0)  # Bias

In [28]:
# Binary Cross-Entropy Loss for scalar
def binary_cross_entropy_loss(prediction, target):
    epsilon = 1e-8  # To prevent log(0)
    prediction = torch.clamp(prediction, epsilon, 1 - epsilon)
    return -(target * torch.log(prediction) + (1 - target) * torch.log(1 - prediction))

In [29]:
# Forward pass
z = w * x + b  # Weighted sum (linear part)
y_pred = torch.sigmoid(z)  # Predicted probability

# Compute binary cross-entropy loss
loss = binary_cross_entropy_loss(y_pred, y)

In [30]:
loss

tensor(6.7012)

In [31]:
# Derivatives:
# 1. dL/d(y_pred): Loss with respect to the prediction (y_pred)
dloss_dy_pred = (y_pred - y)/(y_pred*(1-y_pred))

# 2. dy_pred/dz: Prediction (y_pred) with respect to z (sigmoid derivative)
dy_pred_dz = y_pred * (1 - y_pred)

# 3. dz/dw and dz/db: z with respect to w and b
dz_dw = x  # dz/dw = x
dz_db = 1  # dz/db = 1 (bias contributes directly to z)

dL_dw = dloss_dy_pred * dy_pred_dz * dz_dw
dL_db = dloss_dy_pred * dy_pred_dz * dz_db

In [32]:
print(f"Manual Gradient of loss w.r.t weight (dw): {dL_dw}")
print(f"Manual Gradient of loss w.r.t bias (db): {dL_db}")

Manual Gradient of loss w.r.t weight (dw): 6.691762447357178
Manual Gradient of loss w.r.t bias (db): 0.998770534992218


# 3.Calculating the derivates(gradients) for a neural network using autograd:



In [33]:
x=torch.tensor(6.7)
y=torch.tensor(0.0)

In [34]:
w=torch.tensor(1.0,requires_grad=True)
b=torch.tensor(0.0,requires_grad=True)

In [35]:
w

tensor(1., requires_grad=True)

In [36]:
b

tensor(0., requires_grad=True)

In [37]:
#forward propogation:
z=w*x+b
z

tensor(6.7000, grad_fn=<AddBackward0>)

In [40]:
y_pred=torch.sigmoid(z)
y_pred

tensor(0.9988, grad_fn=<SigmoidBackward0>)

In [41]:
loss=binary_cross_entropy_loss(y_pred,y)
loss

tensor(6.7012, grad_fn=<NegBackward0>)

In [42]:
loss.backward() #-> backward propagation

In [44]:
print(w.grad) #->gradient of w after back propagation
print(b.grad) #->gradient of b after back propagation

tensor(6.6918)
tensor(0.9988)


# 4. Using autograd with vector input tensors:

In [48]:
x=torch.tensor([1.0,2.0,3.0],requires_grad=True)
x

tensor([1., 2., 3.], requires_grad=True)

In [50]:
y=(x**2).mean()
y

tensor(4.6667, grad_fn=<MeanBackward0>)

In [51]:
y.backward()

In [52]:
x.grad

tensor([0.6667, 1.3333, 2.0000])

# 5.Clearing gradients:

In [73]:
x=torch.tensor(2.0,requires_grad=True)
x

tensor(2., requires_grad=True)

In [74]:
y=x**2
y

tensor(4., grad_fn=<PowBackward0>)

In [75]:
y.backward()

In [76]:
x.grad

tensor(4.)

In PyTorch, gradient accumulation means that every time you call backward(), the newly computed gradients are added to the existing values in .grad instead of replacing them. This happens because PyTorch assumes you may want to accumulate gradients over multiple batches or steps. If you don’t clear them, gradients will keep piling up and lead to incorrect weight updates. That’s why during training we usually call optimizer.zero_grad() (or set gradients to zero) before the next backward pass to start fresh.

In [77]:
x.grad.zero_()

tensor(0.)

# 6. Disabling Gradient Tracking:

In PyTorch, gradient tracking refers to the mechanism where PyTorch records all operations performed on tensors that have requires_grad=True. While doing the forward pass, PyTorch builds a computation graph that remembers how each value was produced. When backward() is called, this graph is used to compute gradients using the chain rule. If gradient tracking is disabled (for example, using torch.no_grad()), PyTorch does not build this graph, saving memory and computation, but gradients will not be computed.

In [80]:
x=torch.tensor(2.0,requires_grad=True)
x

tensor(2., requires_grad=True)

In [82]:
y=x**2
y

tensor(4., grad_fn=<PowBackward0>)

In [83]:
y.backward()

In [84]:
x.grad

tensor(4.)

below are three ways to disbale gradient tracking:

In [85]:
#option 1: requires_grad=False
#option 2: detach()
#option 3: torch.no_grad()